In [ ]:
#| default_exp hyperparameter_tuning

## Hyperparameter Tuning

Peshbeen provides two powerful tools for hyperparameter tuning: `hyperopt_tune` and `optuna_tune`. These functions allow you to optimize the hyperparameters of your forecasting models using the `hyperopt` and `optuna` libraries, respectively. Both functions support cross-validation and can be used with any of the forecasting models available in peshbeen.

### hyperopt_tune example for univariate forecasting using machine learning models

In [ ]:
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import hyperopt_tune
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown="ignore")

wales_admissions = load_wales_admissions()
wales_admissions["day_of_week"] = wales_admissions.index.dayofweek
wales_admissions["month"] = wales_admissions.index.month
# split the data into train and test sets
train = wales_admissions[:-30]
test = wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
ml_model = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)

# Define the hyperparameter search space for LightGBM
from hyperopt import hp
from hyperopt.pyll import scope
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
           'top_rate' : hp.quniform('top_rate', 0.05, 0.4, 0.0001),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'lags': hp.choice("lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14],
                                 [1,2,3,4,5,6,7,14,21],
                                 [1,2,3],
                             ]),
                "seed":0,
                "box_cox": hp.uniform("box_cox", 0.0, 4),
                "box_cox_biasadj": hp.choice("box_cox_biasadj", [True, False])}

# Run hyperparameter tuning using hyperopt
best_params, best_lags, other_, _ = hyperopt_tune(model=ml_model, df=train, cv_split=5, step_size=10,
                                        test_size=1, eval_metric=RMSE, eval_num=10,
                                        param_space=lgb_param_space)

print("Best params:", best_params)
print("Best lags:", best_lags)
print("Other info:", other_)

Best params: {'bagging_fraction': 0.666350574241346, 'feature_fraction': 0.9935116794809394, 'lambda_l1': 7.426195112359075, 'lambda_l2': 1.9405235460913972, 'learning_rate': 0.09696236947103197, 'max_depth': 8, 'num_iterations': 132, 'num_leaves': 178, 'other_rate': 0.056, 'seed': 0, 'top_rate': 0.1701}
Best lags: (1, 2, 3, 4, 5, 6, 7)
Other info: {'box_cox': 3.6533929419712443, 'box_cox_biasadj': False}


In [ ]:
# now we can run our model with the best paramaters, best lags and other info such as box_cox and box_cox_biasadj
best_box_cox = other_["box_cox"]
best_box_cox_biasadj = other_["box_cox_biasadj"]

ml_model = ml_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_col='admissions', lags = list(best_lags), box_cox=best_box_cox, box_cox_biasadj=best_box_cox_biasadj,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)
ml_forecasts = ml_model.forecast(H=30, exog=test[cat_variables])
ml_forecasts

array([8850.54431181, 8766.97723554, 8749.64836222, 8917.30187726,
       8884.04610134, 8908.2577349 , 8863.13998023, 8819.41834906,
       8713.12029046, 8698.31737734, 8847.49144705, 8859.13190837,
       8899.30054802, 8877.76454752, 8824.55429231, 8679.25817774,
       8658.105725  , 8829.26846346, 8781.83991167, 8828.6862378 ,
       8832.92581779, 8804.64638798, 8632.39794071, 8611.73610704,
       8705.46017803, 8748.20486497, 8782.73369734, 8783.11016144,
       8745.690588  , 8625.97844142])

### optuna_tune example for univariate forecasting

In [ ]:
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import MAE, RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import optuna_tune
wales_admissions = load_wales_admissions()
wales_admissions["day_of_week"] = wales_admissions.index.dayofweek
wales_admissions["month"] = wales_admissions.index.month
# split the data into train and test sets
train = wales_admissions[:-30]
test = wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
ml_model = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)
# ml_model.data_prep(train)
forecasts = ml_model.forecast(H=30, exog=test[cat_variables])
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "top_rate":          lambda t: t.suggest_float("top_rate", 0.05, 0.4),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "lags":              lambda t: t.suggest_categorical(
                             "lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14],
                                 [1,2,3,4,5,6,7,14,21],
                                 [1,2,3],
                             ]),
                             
    "seed":              lambda t: 0,   # fixed, not sampled
}

best_params, best_lags, other_, _ = optuna_tune(
    model=ml_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=10,
    param_space=lgb_param_space, verbose=False
)
print("Best params:", best_params)
print("Best lags:", best_lags)

[I 2026-05-08 17:16:07,147] A new study created in memory with name: no-name-c807753c-65af-446c-bea4-00857fa05f9a
[I 2026-05-08 17:16:14,100] Trial 0 finished with value: 131.04748621952425 and parameters: {'learning_rate': 0.3421896446799838, 'num_leaves': 138, 'max_depth': 18, 'bagging_fraction': 0.6658111568920178, 'feature_fraction': 0.541434262756582, 'lambda_l2': 0.09640374739570134, 'lambda_l1': 1.4282695341693719, 'top_rate': 0.1939090990429544, 'other_rate': 0.12155184201754614, 'num_iterations': 238, 'lags': [1, 2, 3, 4, 5, 6, 7]}. Best is trial 0 with value: 131.04748621952425.
[I 2026-05-08 17:16:24,825] Trial 1 finished with value: 114.28484431186355 and parameters: {'learning_rate': 0.25054229261166683, 'num_leaves': 47, 'max_depth': 10, 'bagging_fraction': 0.5651716782666025, 'feature_fraction': 0.9527280772921196, 'lambda_l2': 4.184428743952329, 'lambda_l1': 0.31312174820905314, 'top_rate': 0.07967101927150896, 'other_rate': 0.17852531722650528, 'num_iterations': 614, '

Best params: {'learning_rate': 0.13269651854027953, 'num_leaves': 86, 'max_depth': 8, 'bagging_fraction': 0.7787350439702846, 'feature_fraction': 0.9033246340456604, 'lambda_l2': 7.2303202506666615, 'lambda_l1': 5.198902433178746, 'top_rate': 0.3088502650475166, 'other_rate': 0.16984120814048898, 'num_iterations': 507}
Best lags: [1, 4, 7]


### Selecting the best ETS (Error, Trend, Seasonality) model usig `optuna_tune` or `hyperopt_tune`

In [ ]:
from peshbeen.models import ets

ets_param_space = {
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "trend":              lambda t: t.suggest_categorical(
                             "trend", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "seasonal":           lambda t: t.suggest_categorical(
                             "seasonal", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "smoothing_trend":    lambda t: t.suggest_float("smoothing_trend", 0.001, 0.99),
    "smoothing_seasonal": lambda t: t.suggest_float("smoothing_seasonal", 0.001, 0.99),
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "seasonal_periods":              lambda t: 7,   # fixed, not sampled
        "box_cox":           lambda t: t.suggest_float("box_cox", 0.0, 4),
        "box_cox_biasadj":   lambda t: t.suggest_categorical("box_cox_biasadj", [True, False])
}

ets_model = ets(target_col='admissions')
best_params, _, other_, _ = optuna_tune(
    model=ets_model,
    df=train,
    cv_split=4,
    step_size=1,
    test_size=30,
    eval_metric=RMSE,
    eval_num=100,
    param_space=ets_param_space, verbose=False
)
print("Best params:", best_params)
print("Other info:", other_)

[I 2026-05-08 17:17:02,509] A new study created in memory with name: no-name-1770f09e-70c4-490f-beab-15940823fa6b
[I 2026-05-08 17:17:02,540] Trial 0 finished with value: 242.47243509865552 and parameters: {'smoothing_level': 0.4785271999807372, 'trend': None, 'seasonal': 'mul', 'smoothing_trend': 0.3013976418267778, 'smoothing_seasonal': 0.6201088430922946, 'box_cox': 0.5286043858317249, 'box_cox_biasadj': False}. Best is trial 0 with value: 242.47243509865552.
[I 2026-05-08 17:17:02,609] Trial 1 finished with value: 990.4556334662444 and parameters: {'smoothing_level': 0.6854930263749538, 'trend': 'mul', 'seasonal': 'mul', 'smoothing_trend': 0.31401124126409186, 'smoothing_seasonal': 0.05355341869406734, 'box_cox': 3.0283314733190787, 'box_cox_biasadj': False}. Best is trial 0 with value: 242.47243509865552.
[I 2026-05-08 17:17:02,668] Trial 2 finished with value: 2.2097158138497264e+26 and parameters: {'smoothing_level': 0.9097623415221712, 'trend': 'mul', 'seasonal': 'mul', 'smooth

Best params: {'smoothing_level': 0.4382812904485665, 'trend': None, 'seasonal': None, 'smoothing_trend': 0.3847747443582618, 'smoothing_seasonal': 0.5841787810112108}
Other info: {'box_cox': 1.9314681823553548, 'box_cox_biasadj': True}


In [ ]:
# now forecast wit the best parameters
best_params.update(other_)
ets_model = ets(target_col='admissions', **best_params)
ets_model.fit(train)
ets_forecasts = ets_model.forecast(H=30)

In [ ]:
# get cross validation results with the best parameters
cv_df = ets_model.cross_validate(
    df=train,
    cv_split=5,
        step_size=7,
        test_size=30,
        metrics=[RMSE, MAE])
cv_df.head()

,cutoff,index,split,y_true,y_pred
0,2022-12-07,2022-12-07,fold_1,8933,8931.283792
1,2022-12-07,2022-12-08,fold_1,9013,8931.283792
2,2022-12-07,2022-12-09,fold_1,9000,8931.283792
3,2022-12-07,2022-12-10,fold_1,8821,8931.283792
4,2022-12-07,2022-12-11,fold_1,8835,8931.283792


### Selecting the best orders for an ARIMA model using `optuna_tune` or `hyperopt_tune`

In [ ]:
from peshbeen.models import arima
from itertools import product
# Define the hyperparameter search space for ARIMA
p_values = [0, 1, 2, 3]
d_values = [1]
q_values = [0, 1, 2, 3]

# create the list of orders using the product of p, d and q values
orders = list(product(p_values, d_values, q_values))

# Define the hyperparameter search space for seasonal ARIMA
P_values = [0, 1, 2, 3]
D_values = [0, 1]
Q_values = [0, 1, 2, 3]

# create the list of seasonal orders using the product of P, D and Q values
seasonal_orders = list(product(P_values, D_values, Q_values))

# let's define the hyperparameter space for arima using hyperopt
arima_param_space = {
    "order": hp.choice("order", orders),
    "seasonal_order": hp.choice("seasonal_order", seasonal_orders),
    "seasonal_length": 7,
    "box_cox": hp.uniform("box_cox", 0.0, 4),
    "box_cox_biasadj": hp.choice("box_cox_biasadj", [True, False])
}

arima_model = arima(target_col='admissions')
best_params, _, other_, _ = hyperopt_tune(
    model=arima_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=5,
    param_space=arima_param_space
)

### hyperopt_tune example for multivariate forecasting

In [ ]:
from peshbeen.datasets import load_admission_calls
from peshbeen.models import ml_mv_forecaster
from peshbeen.metrics import RMSE
from peshbeen.model_selection import mv_hyperopt_tune
from lightgbm import LGBMRegressor
admission_calls = load_admission_calls()

admission_calls["day_of_week"] = admission_calls.index.dayofweek
admission_calls["month"] = admission_calls.index.month
train = admission_calls[:-30]
test = admission_calls[-30:]

cat_variables = ["day_of_week", "month"]
mv_model = ml_mv_forecaster(model=LGBMRegressor(verbose=-1),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables,
                difference={"admissions": 1, "calls": 1}, categorical_encoder=ohe)
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
           'min_data_in_leaf': scope.int(hp.quniform ('min_data_in_leaf', 5, 100, 1)), 
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'top_k': scope.int(hp.quniform('top_k', 8, 30, 1)),
                "seed":0, 'lags': hp.choice("lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14]])}
best_params, best_lags = mv_hyperopt_tune(model=mv_model, df=train, target_col= "admissions", cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=4,
                                        param_space=lgb_param_space)
print("Best params:", best_params)
print("Best lags:", best_lags)

100%|██████████| 4/4 [00:06<00:00,  1.63s/trial, best loss: 227.86677655884955]
Best params: {'bagging_fraction': 0.6816407076496984, 'feature_fraction': 0.6492594892038641, 'lambda_l1': 3.824135109636244, 'lambda_l2': 7.684885772218308, 'learning_rate': 0.2280098610251414, 'max_depth': 15, 'min_data_in_leaf': 63, 'num_iterations': 504, 'num_leaves': 200, 'other_rate': 0.24100000000000002, 'seed': 0, 'top_k': 27}
Best lags: {'admissions': [1, 2, 3, 4, 5], 'calls': [1, 2, 3, 4, 5]}


In [ ]:
# forecast with the best parameters and best lags
mv_model = ml_mv_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_cols=['admissions', "calls"], lags = best_lags,
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})
mv_model.fit(train)
mv_forecasts = mv_model.forecast(H=30, exog=test[cat_variables])
mv_forecasts

{'admissions': array([7976.0971294 , 7968.58783963, 7879.98838042, 7843.12946402,
        7936.03704634, 7907.53877778, 7985.62499833, 8045.24139756,
        8088.44259739, 8031.64981599, 8051.15245375, 7919.16757969,
        7999.28864143, 8102.38319356, 8064.99090474, 8048.94703133,
        7975.61261953, 7981.27100285, 7910.95417569, 7904.25878917,
        7894.95800544, 7983.09942574, 8063.31627673, 8060.36216988,
        8097.14110756, 8099.67239945, 7952.91836831, 7959.87086905,
        8022.01867843, 8067.95742228]),
 'calls': array([1196.43305729, 1253.91636724, 1199.87095416, 1244.26358435,
        1262.91983329, 1241.74836558, 1244.58054448, 1286.43402398,
        1213.5033088 , 1224.1450858 , 1262.24301881, 1250.14293468,
        1180.7417229 , 1281.34761223, 1242.27991579, 1264.72021051,
        1216.73780551, 1230.86175235, 1273.19320241, 1262.43241218,
        1277.26152547, 1261.61123826, 1314.59525573, 1294.61675002,
        1293.1027092 , 1321.98966503, 1323.54325987, 

### optuna_tune example for multivariate forecasting

In [ ]:
from peshbeen.model_selection import mv_optuna_tune
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "min_data_in_leaf":  lambda t: t.suggest_int("min_data_in_leaf", 5, 100),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "top_k":             lambda t: t.suggest_int("top_k", 8, 30),
    "seed":              lambda t: 0,   # fixed, not sampled
    'lags': lambda t: t.suggest_categorical(
                             "lags", [1,2,3,4,5,
                                [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14]
                             ]),    
}

mv_model = ml_mv_forecaster(model=LGBMRegressor(verbose=-1),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})

best_params, best_lags = mv_optuna_tune(model=mv_model, df=train, target_col= "admissions", cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=4,
                                        param_space=lgb_param_space)

In [ ]:
# forecast with the best parameters and best lags from optuna
mv_model = ml_mv_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_cols=['admissions', "calls"], lags = best_lags,
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})
mv_model.fit(train)
mv_forecasts = mv_model.forecast(H=30, exog=test[cat_variables])
mv_forecasts

{'admissions': array([8081.61582761, 8145.60612625, 8195.69815452, 8174.95203522,
        8133.24326962, 8106.02865958, 7997.86306235, 8070.19121279,
        8209.08299998, 8211.99157305, 8133.83956406, 8128.3562357 ,
        8085.9475194 , 7945.84448369, 7973.82148427, 8144.54249762,
        8217.27620103, 8186.75488321, 8185.39634919, 8091.44544315,
        8101.64343163, 8113.03987282, 8202.07851351, 8224.35724332,
        8284.91388158, 8259.67562671, 8188.18047851, 8048.06639381,
        8054.36624128, 8210.77491789]),
 'calls': array([1233.40748759, 1264.57335053, 1229.21031849, 1250.78804819,
        1247.51753885, 1297.1628157 , 1249.80669766, 1234.0381711 ,
        1282.45496409, 1202.76770185, 1207.20834424, 1186.88617663,
        1274.14456091, 1249.61145261, 1317.22980332, 1281.90732206,
        1279.96585335, 1264.75043671, 1303.57562748, 1274.02694809,
        1302.16408636, 1270.5543115 , 1390.67528644, 1265.15522582,
        1312.31535665, 1292.72770557, 1289.34184394, 